# 1-2회차 | scikit-learn 실습 & Iris 분류

**오늘의 목표**
- 머신러닝이 *무엇*이고 *왜* 필요한지 감 잡기
- scikit-learn을 쓸 준비 (일관된 API 구조)
- Iris 데이터셋의 구조를 파악하고 EDA 실습
- 첫 모델을 학습하고 예측 및 정확도 계산
- K-Fold 교차검증으로 안정적인 성능 측정
- GridSearchCV로 하이퍼파라미터 자동 탐색

---
## Part 1. 머신러닝과 scikit-learn 소개

## 머신러닝이란?
사람이 규칙을 직접 코딩하지 않고, **데이터로부터 규칙을 학습해** 새로운 데이터를 예측/분류하는 방법입니다.

- 규칙 프로그래밍: `"제목에 '광고' 포함 → 스팸"` 처럼 사람이 규칙을 작성
- 머신러닝: 과거 수십만 건의 메일 데이터를 보고 **스스로** 스팸 패턴을 학습

## 머신러닝의 주요 종류
1) **지도학습(Supervised)** — 정답(y)이 있음
   - **분류**: 스팸/정상, 양성/음성 (라벨 예측)
   - **회귀**: 집값/매출/보험료 (숫자 예측)
2) **비지도학습(Unsupervised)** — 정답 없음
   - **군집화**: 고객 세그멘테이션
   - **차원축소**: 시각화/압축(PCA 등)
3) **강화학습(Reinforcement)** — 보상 기반 의사결정(게임/로보틱스)

## 용어 정리
- **Feature(특성, X)**: 모델이 입력으로 받는 정보(예: 전재산, 혈액형, 건강검진기록).
- **Label/Target(정답, y)**: 모델이 맞추려는 값(예: 생존 여부, 암 양성/음성).
- **Estimator(추정기)**: `fit/predict`를 제공하는 모델/전처리기(예: LogisticRegression, StandardScaler).
- **Transformer(변환기)**: `fit/transform`으로 데이터를 바꾸는 전처리기(예: StandardScaler, OneHotEncoder).
- **Pipeline**: 전처리 + 모델을 한 줄로 묶어 "훈련에만 fit, 시험엔 transform"을 자동으로 지키게 하는 래퍼.
- **Metric(평가지표)**: 성능을 숫자로 표현(분류: Accuracy/F1/ROC-AUC, 회귀: RMSE/MAE/R²).

## 생활 속 사례
- 스팸메일 필터
- 추천 시스템(넷플릭스/유튜브)
- 질병 진단 보조(당뇨/암)
- 자율주행 인식

메시지: *여러분은 이미 매일 ML을 사용하고 있습니다.*

## 머신러닝 프로젝트 흐름
1. **데이터 준비**
2. **학습/시험 데이터 분리** *(Train/Test)*
3. **모델 학습** `fit(X_train, y_train)`
4. **예측** `predict(X_test)`
5. **평가** *(metrics: 분류=정확도/F1/ROC 등, 회귀=MAE/RMSE/R²)*
6. **개선** *(튜닝, 전처리, 더 나은 모델)*

> **비유**: 같은 문제로 공부하고 같은 문제로 시험 보면 **성적이 뻥튀기**됨
> 그래서 **Train/Test**를 나눠 *처음 보는 문제*로 실력을 확인해야 함

## scikit-learn 이란?
파이썬의 **머신러닝 표준 라이브러리**(2007 시작). 교육/실무 모두에서 가장 널리 쓰임

- **일관된 API(사용방법)**: `fit → predict → score` (모델이 달라도 동일한 사용법)
- 알고리즘(분류/회귀/군집화), **전처리**, **모델 선택(CV/GridSearch)**, **평가 지표**까지 원스톱
- 문서/예제가 풍부하고 커뮤니티가 큼

**오늘의 포인트**: *모델(알고리즘)을 바꿔도 `fit→predict→score` 패턴은 같음!*

In [13]:
##설치
!pip install scikit-learn 
 
#!pip install -U scikit-learn ##업그레이드

##앞에 ! 대신 % 기호를 붙여 실행하면, 현재 노트북 커널이 구동 중인 정확한 파이썬 환경을 찾아가서 라이브러리를 강제로 설치해 줍니다)
# %pip install scikit-learn 


## 환경/버전 확인
아래 셀을 실행해 현재 환경 버전을 기록해 둡니다.

나중에 같은 결과를 재현하고 환경 차이로 인한 오류를 줄이기 위해 버전을 기록합니다.

In [14]:
import sklearn
import numpy as np, pandas as pd
import matplotlib, seaborn

print(f'sklearn: {sklearn.__version__}')
print(f'numpy: {np.__version__}')
print(f'pandas: {pd.__version__}')
print(f'matplotlib: {matplotlib.__version__}')
print(f'seaborn: {seaborn.__version__}')

sklearn: 1.9.0
numpy: 2.3.5
pandas: 3.0.5
matplotlib: 3.10.6
seaborn: 0.13.2


## 한 장 요약 (오늘 이것만 기억)
1) ML = 코딩이 아니라 **데이터로 규칙을 학습**
2) 모든 모델은 **`fit → predict → score`**
3) **Train/Test**를 나눠야 성적이 뻥튀기되지 않음
4) 전처리는 **훈련에만 fit**, 시험엔 transform → **Pipeline**
5) 분류는 **Accuracy+F1/ROC-AUC**, 회귀는 **RMSE/MAE/R²**
6) 실무는 **교차검증(K-Fold)**으로 평균 성능을 본다

---
## Part 2. Iris 데이터셋 이해 (EDA)

**목표**
- scikit-learn 내장 데이터셋(Bunch)의 공통 구조를 이해
- Iris 데이터셋의 피처(꽃받침/꽃잎 길이·너비)와 타깃(3개 품종) 파악
- 간단 EDA로 클래스별 차이를 데이터로 확인

### sklearn 내장 데이터셋 구조 (Bunch 객체)

`load_iris()` 등으로 데이터를 로드하면 딕셔너리와 유사한 **Bunch 객체**가 반환

키는 보통 `data`, `target`, `target_names`, `feature_names`, `DESCR`로 구성

| 키 | 내용 |
|----|------|
| `data` | 특성 데이터 (숫자 배열) |
| `target` | 정답 라벨 (숫자 배열) |
| `target_names` | 정답 라벨의 이름 |
| `feature_names` | 특성(컬럼)의 이름 |
| `DESCR` | 데이터셋 설명문 |

In [15]:
from sklearn.datasets import load_iris

iris_data = load_iris()
iris_data

{'data': array([[5.1, 3.5, 1.4, 0.2],
        [4.9, 3. , 1.4, 0.2],
        [4.7, 3.2, 1.3, 0.2],
        [4.6, 3.1, 1.5, 0.2],
        [5. , 3.6, 1.4, 0.2],
        [5.4, 3.9, 1.7, 0.4],
        [4.6, 3.4, 1.4, 0.3],
        [5. , 3.4, 1.5, 0.2],
        [4.4, 2.9, 1.4, 0.2],
        [4.9, 3.1, 1.5, 0.1],
        [5.4, 3.7, 1.5, 0.2],
        [4.8, 3.4, 1.6, 0.2],
        [4.8, 3. , 1.4, 0.1],
        [4.3, 3. , 1.1, 0.1],
        [5.8, 4. , 1.2, 0.2],
        [5.7, 4.4, 1.5, 0.4],
        [5.4, 3.9, 1.3, 0.4],
        [5.1, 3.5, 1.4, 0.3],
        [5.7, 3.8, 1.7, 0.3],
        [5.1, 3.8, 1.5, 0.3],
        [5.4, 3.4, 1.7, 0.2],
        [5.1, 3.7, 1.5, 0.4],
        [4.6, 3.6, 1. , 0.2],
        [5.1, 3.3, 1.7, 0.5],
        [4.8, 3.4, 1.9, 0.2],
        [5. , 3. , 1.6, 0.2],
        [5. , 3.4, 1.6, 0.4],
        [5.2, 3.5, 1.5, 0.2],
        [5.2, 3.4, 1.4, 0.2],
        [4.7, 3.2, 1.6, 0.2],
        [4.8, 3.1, 1.6, 0.2],
        [5.4, 3.4, 1.5, 0.4],
        [5.2, 4.1, 1.5, 0.1],
  

In [16]:
# Bunch 객체의 키 확인 — 딕셔너리처럼 접근 가능
print(f"키 목록: {list(iris_data.keys())}")

키 목록: ['data', 'target', 'frame', 'target_names', 'DESCR', 'feature_names', 'filename', 'data_module']


In [17]:
print(f"feature_names: {iris_data.feature_names}")
print(f"target_names : {iris_data.target_names}")
print(f"data shape   : {iris_data.data.shape}")
print(f"target shape : {iris_data.target.shape}")

feature_names: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
target_names : ['setosa' 'versicolor' 'virginica']
data shape   : (150, 4)
target shape : (150,)


### 피처(특성)와 타깃(품종) 의미
- **sepal length (cm)**: 꽃받침 길이
- **sepal width  (cm)**: 꽃받침 너비
- **petal length (cm)**: 꽃잎 길이
- **petal width  (cm)**: 꽃잎 너비

타깃(label)
- 0 = setosa
- 1 = versicolor
- 2 = virginica


### DataFrame 만들기
판다스 DataFrame으로 변환하면 데이터를 훨씬 편하게 다룰 수 있습니다.

In [18]:
import pandas as pd

iris = load_iris()

iris_data = iris.data

iris_label = iris.target
print('iris target값:', iris_label)
print('iris target명:', iris.target_names)

iris_df = pd.DataFrame(data=iris_data, columns=iris.feature_names)
iris_df['label'] = iris.target
iris_df.head(3)

iris target값: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2]
iris target명: ['setosa' 'versicolor' 'virginica']


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),label
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0


In [19]:
# import pandas as pd
#
# df = pd.DataFrame(iris_data.data, columns=iris_data.feature_names)
# df["label"] = iris_data.target
# label_map = {i: name for i, name in enumerate(iris_data.target_names)}
# df["species"] = df["label"].map(label_map)
# df.head()
# list(label_map)

---
### [실습] 데이터 구조 파악 체크리스트

> 새로운 데이터를 만나면 **가장 먼저** 해야 할 것들입니다.
> 판다스에서 배운 것들을 여기서 바로 활용합니다!

In [20]:
# 1단계: 데이터 크기 확인 — 몇 행, 몇 열?

In [21]:
# 2단계: 각 컬럼의 데이터 타입 확인
# 숫자(float64, int64)인지, 문자(object)인지에 따라 전처리 방법이 달라짐

In [22]:
# 3단계: 데이터 전체 요약 — info()는 타입, 결측치, 메모리까지 한번에!

---
### [실습] 결측치(Missing Values) 확인

> **결측치** = 비어있는 데이터 (NaN).  
> 결측치가 있으면 모델이 학습할 수 없거나 성능이 떨어짐
> 실무 데이터에서는 결측치가 매우 흔하므로, 항상 확인하는 습관이 중요함

In [23]:
# 컬럼별 결측치 수 확인

# Iris는 결측치가 없는 깨끗한 데이터임(미끼임!)
# 실무 데이터에서는 거의 반드시 결측치가 존재함

> **결측치가 있다면?**
> - 해당 행을 삭제: `df.dropna()`
> - 평균값으로 채우기: `df.fillna(df.mean())`
> - 중앙값으로 채우기: `df.fillna(df.median())`
> - sklearn의 `SimpleImputer` 사용 (나중에 배움)
>
> 어떤 방법을 쓸지는 **데이터의 특성**에 따라 다름₩.

---
### [실습] Target(정답) 분포 확인

> 클래스별 데이터 개수가 **균등한지** 확인하는 것이 중요합니다.  
> 한쪽 클래스만 많으면 모델이 편향될 수 있습니다. ("불균형 데이터" 문제)

In [24]:
# Target 분포 확인 — value_counts()로 각 클래스의 개수 세기

# 각 품종이 50개씩, 33.3%로 완벽하게 균등! → 좋은 데이터

In [11]:
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'AppleGothic'   # 맥
# plt.rcParams['font.family'] = 'Malgun Gothic'  # 윈도우
plt.rcParams['axes.unicode_minus'] = False # 한글 폰트 사용 시, 마이너스 글자가 깨지는 현상 해결

In [25]:
# # Target 분포 시각화
# import matplotlib.pyplot as plt

# fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# # 막대 그래프
# df["species"].value_counts().plot(kind="bar", ax=axes[0], color=["#4a9eff", "#ff6b6b", "#51cf66"])
# axes[0].set_title("품종별 데이터 수")
# axes[0].set_ylabel("개수")

# # 파이 차트 — 비율 확인에 직관적
# df["species"].value_counts().plot(kind="pie", ax=axes[1], autopct="%1.1f%%",
#                                    colors=["#4a9eff", "#ff6b6b", "#51cf66"])
# axes[1].set_title("품종별 비율")
# axes[1].set_ylabel("")

# plt.tight_layout()
# plt.show()

# Target 분포 시각화
import plotly.express as px
df_species = df['species'].value_counts().reset_index()

# 막대 그래프
fig_bar = px.bar(df_species, x='species', y='count',
                 title='품종별 데이터 수', color='species')
fig_bar.show()

# 파이 차트-비율 확인에 직관적
fig_pie = px.pie(df_species, names='species', values='count', title='품종별 비율', hole=0)
fig_pie.update_traces(textposition='inside', textinfo='percent+label')
fig_pie.show()

NameError: name 'df' is not defined

---
### 기본 통계 확인

In [ ]:
# describe(): count, mean, std, min, 25%, 50%, 75%, max 한번에
df.describe()

> **describe()에서 봐야 할 포인트:**
> - `count`: 결측치가 있으면 컬럼마다 count가 다름
> - `mean`과 `std`: 데이터의 중심과 퍼진 정도
> - `min`과 `max`: 이상치가 있는지 확인
> - `25%`, `50%`, `75%`: 데이터의 분포 형태 파악

In [ ]:
df.groupby("species")[iris_data.feature_names].mean().round(2)

> **관찰:**
> - `petal length/width`: 품종 간 차이가 매우 뚜렷 → 분류에 유리한 피처!
> - `sepal width`: 품종 간 차이가 상대적으로 작음 → 분류에 덜 유리

---
### 시각화 EDA

> 모델을 돌리기 전에 데이터가 어떻게 생겼는지 **눈으로** 확인하셈
> 시각화를 통해 예측 난이도를 미리 가늠하고, 어떤 알고리즘이 적합할지 판단할 수 있음

In [ ]:
import plotly.express as px

fig = px.scatter(
    df,
    x="petal length (cm)",
    y="petal width (cm)",
    color="species",
    title="Iris: 꽃잎 길이 대 너비 (Petal)"
)
fig.show()

In [ ]:
fig = px.scatter(
    df,
    x="sepal length (cm)",
    y="sepal width (cm)",
    color="species",
    title="Iris: 꽃받침 길이 대 너비 (Sepal)"
)
fig.show()

> **산점도에서 읽어내야 할 것:**
> - 꽃잎(petal) 그래프: setosa가 완전히 분리됨 → 쉽게 구분 가능
> - 꽃받침(sepal) 그래프: 세 품종이 많이 겹침 → 이 피처만으로는 분류 어려움
> - versicolor와 virginica는 꽃잎 그래프에서도 일부 겹침 → 모델이 헷갈릴 수 있는 구간

In [ ]:
# sns.histplot(
#     data=df, x="petal length (cm)", hue="species",
#     element="step", stat="density", common_norm=False
# )
# plt.title("Iris: Petal length distribution by species")
# plt.show()

# 히스토그램: 특정 피처의 분포를 품종별로 비교
fig = px.histogram(
    df,
    x="petal length (cm)",
    color="species",
    barmode="overlay",
    nbins=20,
    title="Iris: 꽃잎 길이(Petal Length) 분포"
)
fig.update_traces(opacity=0.6)
fig.show()

In [ ]:
# sns.histplot(
#     data=df, x="sepal length (cm)", hue="species",
#     element="step", stat="density", common_norm=False
# )
# plt.title("Iris: Sepal length distribution by species")
# plt.show()

fig = px.histogram(
    df,
    x="sepal length (cm)",
    color="species",
    barmode="overlay",
    nbins=20,
    title="Iris"
)
fig.update_traces(opacity=0.6)
fig.show()

---
### [실습] 전체 피처 관계 한눈에 보기 (Pairplot)

> 모든 피처 조합의 산점도를 한 번에 그립니다.  
> 어떤 피처 조합이 분류에 유리한지 **한 장**으로 파악할 수 있습니다.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# pairplot: 모든 피처 조합의 산점도 + 대각선에 히스토그램
sns.pairplot(
    df,
    hue="species",
    diag_kind="hist", # 대각선 히스토그램으로 채워짐
    vars=iris_data.feature_names,
    palette="Set1"
)
plt.suptitle("Iris 전체 피처 Pairplot", y=1.02)
plt.show()

> **Pairplot 읽는 법:**
> - 대각선: 각 피처의 히스토그램 (분포)
> - 대각선 아래/위: 두 피처 조합의 산점도
> - 색이 잘 분리되는 조합 = 분류에 유리한 피처 조합
> - petal length + petal width 조합이 가장 잘 분리됨!

In [ ]:
# 박스플롯: 품종별 피처 분포 비교
plt.figure(figsize=(12, 4))

for i, feature in enumerate(iris_data.feature_names):
    plt.subplot(1, 4, i + 1)
    sns.boxplot(data=df, x="species", y=feature)
    plt.title(feature.replace(" (cm)", ""))
    if i >= 0:
        plt.ylabel("")

plt.suptitle("품종별 피처 분포 (Boxplot)", y=1.02)
plt.tight_layout()
plt.show()

> **박스플롯 읽는 법:**
> - 상자 = 데이터의 25%~75% 구간 (IQR)
> - 상자 안의 선 = 중앙값 (50%)
> - 수염 = 상자에서 1.5 x IQR 범위
> - 수염 밖의 점 = 이상치(outlier)
> - 상자가 겹치지 않는 피처 → 분류에 유리!

---
### [실습] 상관관계 확인

> 피처들 간의 관계를 숫자로 확인합니다.  
> **상관계수**: 1에 가까우면 강한 양의 상관, -1에 가까우면 강한 음의 상관, 0이면 관계 없음

In [ ]:
# 수치 피처만 추출해서 상관행렬 계산
corr = df[iris_data.feature_names].corr().round(2)
print("피처 간 상관계수:")
print(corr)

In [ ]:
# # 상관행렬 히트맵 — 색으로 직관적 확인
# plt.figure(figsize=(7, 5))
# sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, vmin=-1, vmax=1,
#             square=True, linewidths=1)
# plt.title("피처 간 상관관계 (Correlation Matrix)")
# plt.show()

# 상관행렬 히트맵- 색으로 직관적 확인
fig=px.imshow(corr, text_auto=True, color_continuous_scale='RdBu_r')
fig.update_layout(title='피처 간 상관관계 (Correlation Matrix)')
fig.show()

> **상관관계에서 읽어내야 할 것:**
> - `petal length`와 `petal width`의 상관계수 ≈ 0.96 → 매우 강한 양의 상관 (함께 커짐)
> - `sepal width`는 다른 피처들과 상관이 낮음 → 독립적인 정보를 제공
> - 상관이 너무 높은 피처들은 중복 정보일 수 있음 → 나중에 피처 선택/제거 고려

In [ ]:
# 품종별 평균 특성 히트맵
group_means = df.groupby("species")[iris_data.feature_names].mean()

fig = px.imshow(
    group_means,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="Blues",
    title="품종별 평균 특성 (Heatmap)"
)

fig.update_xaxes(title="특성(feature)")
fig.update_yaxes(title="품종(species)")

fig.show()

### EDA 요약
- **setosa**는 꽃잎 길이/너비가 매우 작아 쉽게 구분
- **versicolor**와 **virginica**는 꽃잎 분포가 일부 겹쳐 모델이 헷갈릴 수 있음
- **petal length / petal width**가 분류에 가장 유리한 피처
- 결측치 없음, 클래스 균등 → 깨끗한 교육용 데이터

---
## EDA 체크리스트 정리

새로운 데이터셋을 만나면 이 순서로:

| 순서 | 코드 | 확인하는 것 |
|------|------|-------------|
| 1 | `df.shape` | 몇 행, 몇 열? |
| 2 | `df.dtypes` / `df.info()` | 각 컬럼의 데이터 타입 |
| 3 | `df.isnull().sum()` | 결측치 있나? |
| 4 | `df.describe()` | 기본 통계 (평균, 표준편차, 최솟값, 최댓값) |
| 5 | `df["target"].value_counts()` | 클래스 분포 균등한가? |
| 6 | 산점도 / 박스플롯 / 히스토그램 | 시각적 패턴 확인 |
| 7 | `df.corr()` + 히트맵 | 피처 간 상관관계 |

> 이 과정 없이 모델을 돌리면, **왜 성능이 안 나오는지 알 수 없습니다.**

---
## 참고: sklearn 내장 데이터셋 더 살펴보기

Iris 외에도 sklearn에는 여러 내장 데이터셋이 있습니다.  
모두 동일한 Bunch 구조를 가지고 있어서, 위에서 배운 방법을 그대로 적용할 수 있습니다.

In [ ]:
from sklearn.datasets import load_breast_cancer, load_diabetes

bc = load_breast_cancer()    # 유방암 이진 분류 (악성/양성)
dia = load_diabetes()        # 당뇨병 회귀 (수치 예측)

print(f"Breast Cancer: {bc.data.shape}, 분류={bc.target_names}")
print(f"Diabetes     : {dia.data.shape}, 회귀 (타겟=수치)")

# Bunch 구조가 동일! → 키만 바꾸면 같은 코드로 분석 가능
print(f"\nBreast Cancer keys: {list(bc.keys())}")
print(f"Diabetes keys     : {list(dia.keys())}")

---
## 모델은 이렇게 사용됩니다 (미리보기)

머신러닝 모델은 보통 이 3단계로 사용됩니다.

1. **fit** — 데이터로부터 규칙을 학습
2. **predict** — 처음 보는 데이터 예측
3. **score** — 예측이 얼마나 맞았는지 평가

모델이 달라져도 이 패턴은 **동일**합니다!

In [ ]:
# 맛보기: 모델 사용 패턴 (다음 시간에 자세히!)
from sklearn.tree import DecisionTreeClassifier

# 1. 모델 생성
model = DecisionTreeClassifier()

# 2. 학습 (fit) — 데이터의 패턴을 학습
# model.fit(X_train, y_train)

# 3. 예측 (predict) — 새로운 데이터로 예측
# y_pred = model.predict(X_test)

# 4. 평가 (score) — 정확도 확인
# accuracy = model.score(X_test, y_test)

print("다음 시간에 직접 실행해 봅니다!")

> 다음 시간에는 이 과정을 **직접 실습**해봅니다.

---
## 다음 시간 예고

**2회차: 첫 모델 & 모델 선택 기초**

- **첫 번째 모델 실습**: DecisionTree로 fit → predict → score 체험
- **모델 교체**: LogisticRegression, KNN으로 바꿔보기
- **Train/Test Split**: 왜 나눠야 하는지 직접 확인 (accuracy 1.0 착시)
- **교차검증 (K-Fold)**: 한 번 나누기는 운에 좌우 → 여러 번 나눠 평균
- **cross_val_score**: 한 줄로 교차검증 실행
- **GridSearchCV**: 하이퍼파라미터 후보를 공정하게 비교

> 오늘 EDA로 데이터를 파악했으니,  
> 다음엔 **모델을 학습하고 평가하는 전체 흐름**을 익힙니다.